# 05 - Phân tích lỗi

In [11]:
import re, sys, json
from pathlib import Path
from math import comb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)

from src.config import LABELS, DATA_DIR, RESULTS_METRICS, RESULTS_PREDICTIONS

MODELS  = ["baseline", "phobert", "visobert"]
CLASSES = [LABELS[i] for i in sorted(LABELS)]
train = pd.read_csv(DATA_DIR / "train.csv"); val = pd.read_csv(DATA_DIR / "val.csv"); test = pd.read_csv(DATA_DIR / "test.csv")
P = {m: pd.read_csv(RESULTS_PREDICTIONS / f"{m}_test.csv") for m in MODELS}
V = {m: pd.read_csv(RESULTS_PREDICTIONS / f"{m}_val.csv")  for m in MODELS}
R = {m: json.load(open(RESULTS_METRICS / f"{m}.json", encoding="utf-8")) for m in MODELS}
y = test["label_id"].values
assert all((P[m].true_id.values == y).all() for m in MODELS), "dự đoán không khớp test.csv"
print("Đã nạp:", {k: len(v) for k, v in {"train": train, "val": val, "test": test}.items()})

Đã nạp: {'train': 24216, 'val': 3003, 'test': 2985}


In [12]:
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 200)   # đọc được nguyên câu lỗi
from preprocess import TEENCODE, EDGE_PUNCT, EMOJI_RE, HOMOGLYPH   # từ điển/regex dùng chung với bước tiền xử lý

CANDIDATES = ["baseline", "phobert", "visobert"]
MODELS = [m for m in CANDIDATES if (RESULTS_PREDICTIONS / f"{m}_test.csv").exists()]
print("Mô hình đã có dự đoán:", MODELS)
assert MODELS, "Chưa có file dự đoán nào trong results/predictions/"

test = pd.read_csv(DATA_DIR / "test.csv")
P = {}
for m in MODELS:
    p = pd.read_csv(RESULTS_PREDICTIONS / f"{m}_test.csv")
    
    # file dự đoán chỉ lưu văn bản đã xử lý (không có id/câu gốc) -> ghép theo thứ tự dòng và kiểm tra khớp trước khi dùng
    assert len(p) == len(test), f"{m}: số dòng dự đoán ({len(p)}) khác test.csv ({len(test)}) - có thể đã chạy với --limit"
    assert (p["true_id"].values == test["label_id"].values).all(), f"{m}: nhãn thật không khớp test.csv"
    assert (p["text"].fillna("").values == test[text_col(m)].fillna("").values).all(), f"{m}: văn bản không khớp text_{m}"
    
    # file dự đoán mới đã có sẵn id/free_text; file cũ thì lấy từ test.csv (đã kiểm khớp ở trên)
    if "id" not in p.columns:
        p["id"] = test["id"].values
    if "free_text" not in p.columns:
        p["free_text"] = test["free_text"].values
    assert (p["id"].values == test["id"].values).all(), f"{m}: id không khớp test.csv"
    P[m] = p
print("Đã ghép xong, thứ tự dòng khớp với test.csv.")

Mô hình đã có dự đoán: ['baseline', 'phobert', 'visobert']
Đã ghép xong, thứ tự dòng khớp với test.csv.


## 1. So sánh các mô hình

In [13]:
from src.compare import build_summary
display(build_summary())  

,model,val_macro_f1,test_macro_f1,test_weight_f1,test_acc,test_f1_BÌNH THƯỜNG,test_f1_XÚC PHẠM,test_f1_THÙ GHÉT
2,visobert,0.6936,0.6741,0.8695,0.8687,0.9349,0.4581,0.6293
1,phobert,0.6896,0.6912,0.8725,0.8660,0.9332,0.4884,0.6519
0,baseline,0.6771,0.6705,0.8745,0.8777,0.9425,0.4372,0.6317


In [23]:
for m in ["phobert", "visobert"]:
    h = pd.DataFrame(R[m]["extra"]["history"]); h["val F1 so với epoch tốt nhất"] = (h.val_macro_f1 - h.val_macro_f1.max()).round(4)
    print(f"{m} (tốt nhất: epoch {R[m]['extra']['best_epoch']})")
    display(h)

phobert (tốt nhất: epoch 2)


,epoch,train_loss,val_macro_f1,val F1 so với epoch tốt nhất
0,1,0.6828,0.6542,-0.0354
1,2,0.4887,0.6896,0.0000
2,3,0.3653,0.6763,-0.0133
3,4,0.2856,0.6703,-0.0193


visobert (tốt nhất: epoch 4)


,epoch,train_loss,val_macro_f1,val F1 so với epoch tốt nhất
0,1,0.8613,0.6051,-0.0885
1,2,0.5953,0.6525,-0.0411
2,3,0.3948,0.6707,-0.0229
3,4,0.2289,0.6936,0.0000
4,5,0.1274,0.6912,-0.0024
5,6,0.0683,0.6914,-0.0022
6,7,0.0319,0.6891,-0.0045


**overfit của PhoBERT:** từ epoch 2 sang 3 – 4, `train_loss` tiếp tục giảm (0,489 → 0,365 → 0,286) nhưng `val macro-F1` *giảm* (0,690 → 0,676 → 0,670). Mô hình bắt đầu ghi nhớ tập train, nên mô hình sẽ dừng sớm (early stopping)

**ViSoBERT ghi nhớ dữ liệu train:** `train_loss` giảm tới **0,032** ở epoch 7 trong khi val macro-F1 chỉ dao động 0,689 – 0,694 sau epoch 4. Loss gần 0 nghĩa là mô hình đã gần như thuộc lòng train

## 2. Lỗi theo cặp nhãn (thật - đoán)

In [15]:
pairs_dict = {}
for m in MODELS:
    # Lọc ra các câu dự đoán sai và đếm số lượng theo cặp (true, pred)
    incorrect_df = P[m][~P[m].correct]
    pairs_dict[m] = incorrect_df.groupby(["true_label", "pred_label"]).size()

pairs = pd.DataFrame(pairs_dict).fillna(0).astype(int)
display(pairs)

total_errors = {}
for m in MODELS:
    total_errors[m] = int((~P[m].correct).sum())

print("Tổng số lỗi:", total_errors, "trên", len(test), "câu")

baseline  phobert  visobert
true_label  pred_label                              
BÌNH THƯỜNG THÙ GHÉT           71      100        82
            XÚC PHẠM           52      109        84
THÙ GHÉT    BÌNH THƯỜNG        88       57        90
            XÚC PHẠM           33       49        28
XÚC PHẠM    BÌNH THƯỜNG        74       56        63
            THÙ GHÉT           47       29        45

Tổng số lỗi: {'baseline': 365, 'phobert': 400, 'visobert': 392} trên 2985 câu


Test có 2458 câu bình thường và 527 câu độc hại

- False positive: câu BÌNH THƯỜNG bị đoán nhầm thành ĐỘC HẠI -> báo động giả
- False negative: câu DỘC HẠI nhưng đoán thành BÌNH THƯỜNG -> bỏ sót

|Mô hình|Báo động giả|FPR| Bỏ sót| FNR|
|---|---|---|---|---|
|Baseline|123|5%| 162| 30.7%|
|PhoBERT| 209 | 8.5%| 113| 21.4%|
|VisoBERT| 166|6.8%|153|29%|

- baseline: SVM dựa vào từ khóa, câu độc hại không chứa từ khóa quen thuộc thì không có đặc trưng -> đễ nhầm về bình thường
- phobert: loss có trọng số lớp đẩy ranh giới về phía lớp hiếm. nhìn ngữ cảnh ở những câu dùng từ thô tục nhưng mang ý đùa -> dễ đẩy về độc hại
- visobert: ở giữa

## 3. Lỗi và độ tin cậy

In [16]:
rows = []
for m, p in P.items():
    err = p[~p.correct]
    rows.append({"mô hình": m, "số lỗi": len(err),
                 "conf TB (đúng)": round(p[p.correct].confidence.mean(), 3),
                 "conf TB (sai)": round(err.confidence.mean(), 3),
                 "lỗi có conf > 0.8": int((err.confidence > 0.8).sum()),
                 "% lỗi có conf > 0.8": round((err.confidence > 0.8).mean() * 100, 1)})
display(pd.DataFrame(rows))

,mô hình,số lỗi,conf TB (đúng),conf TB (sai),lỗi có conf > 0.8,% lỗi có conf > 0.8
0,baseline,365,0.911,0.637,113,31.0
1,phobert,400,0.939,0.749,190,47.5
2,visobert,392,0.973,0.857,276,70.4


## 4. Xem mẫu lỗi
Mỗi cặp nhãn lỗi nhiều nhất hiển thị `N_SAMPLE` câu ngẫu nhiên (seed cố định)

In [17]:
FOCUS = MODELS[-1]    
N_SAMPLE = 6

err = P[FOCUS][~P[FOCUS].correct]
top_pairs = err.groupby(["true_label", "pred_label"]).size().sort_values(ascending=False).index[:4]
for t, pr in top_pairs:
    sub = err[(err.true_label == t) & (err.pred_label == pr)]
    print(f"\n  [{FOCUS}] Thật: {t}  ->  Đoán: {pr}  ({len(sub)} câu)")
    display(sub.sample(min(N_SAMPLE, len(sub)), random_state=42)[["free_text", "confidence"]])


  [visobert] Thật: THÙ GHÉT  ->  Đoán: BÌNH THƯỜNG  (90 câu)


,free_text,confidence
1367,"giác ngộ đi , tp HCM nhé , bác giết SG rồi 😆😆😆😆",0.9968
878,Cả nước chống dịch ai cũng như ông này chắc thành Vũ Hán hai,0.9860
1892,"Mày chửi LQ xong bị thằng âm binh 11 nó lừa cho hơn trăm triệu, nhục mặt, gậy mày đập vào mặt mày ,đáng đời, chửi người ta làm sao mày gánh hết.",0.5402
2448,"Ở indonesia đã có tàu điện ngầm rồi, ở Đông Lào đợi mãi không có",0.9932
26,ở Việt Nam rất khó...vì lý do...( ai sẽ là người đứng lên chỉ mặt người đàn áp...vì lỗi do thằng đánh máy...thằng đọc hay thằng ký không có tội )...ví dụ thực tế...1 nhóm người tụ tập kêu gọi hòa ...,0.8941
945,Cai mat nha tien tri Tran Dan.nhin giong cai phau cau ga cong nghiep wa mn oi,0.9959



  [visobert] Thật: BÌNH THƯỜNG  ->  Đoán: XÚC PHẠM  (84 câu)


,free_text,confidence
2559,Nói hay làm giở....,0.9796
28,vl tên ạ =)),0.9974
2048,Mãnh thú nhìn sợ vcl :<,0.9822
758,Chặt bỏ chứ để làm gì,0.7040
452,uissss uissss. Ghéppo hàiii vaiiil cácc ôngg ạ = )).,0.9914
2437,"Tu vi của Bác Trần đã đạt đến đỉnh cao của Giác ngộ. Bác ngộ ra được là : Thấy zậy không phải zậy, nghe zậy không phải zậy, biết zậy không phải zậy, đế vương cũng cát bụi, thường dân cũng cát bụi,...",0.5088



  [visobert] Thật: BÌNH THƯỜNG  ->  Đoán: THÙ GHÉT  (82 câu)


,free_text,confidence
1138,🄳🅄 🅃🅁🄴🄽🄳 🄶🄷🄴 🄷🄾🄽🄶 ?,0.4970
59,Đả cẩu bổng pháp của tổng chủ đang triển khai vô cùng lợi hại 😤,0.9127
778,"chúng nó mà biết tính cho dân thì đã ko pháo bông vớ vẫn, tượng đái lung tung và hàng đống công trình ngàn tỷ đắp chiếu...! tuy nhiên chúng lại tính cho chúng rất giỏi ! thằng nào cũng lương có hơ...",0.7986
1308,Hay vai lon anh à,0.9392
678,Ong Tran Dan lam Tong Thong Viet Nam cho dan nga ngua dfo. Mot la gia 2...... Gia roi de cho lop tre no len nam quyen dNG CHUNG NO DU NHAN TAI VE GIUP DO XAY D7UNG vIET nAM.,0.7436
870,"Tục ngữ VN có câu: ""Bói ra ma ,quét nhà ra rác ""bác Dần à, bác nổ vui thật trúng trật mặc kệ.",0.8403



  [visobert] Thật: XÚC PHẠM  ->  Đoán: BÌNH THƯỜNG  (63 câu)


,free_text,confidence
2931,Mất trinh =)))),0.9974
2834,Bởi dừa lờ t lắm cho chừa cái tật chơi dọng cha dọng mẹ rồi bị chặt :)))))),0.9926
2,"Sao tụi bây toàn chửi nhau mà ko thấy làm gì hết,cao lông nó chưa hay nói cho vui",0.9719
1993,Cuộc sống cú vl =)),0.5921
166,Sờ chim a ơu,0.9810
1664,Âm Binh 🙂,0.9963


In [18]:
# câu tất cả mô hình cùng đoán sai: thường là nhãn nhiễu hoặc câu thật sự khó
if len(MODELS) > 1:
    wrong_all = np.logical_and.reduce([~P[m].correct.values for m in MODELS])
    print(f"{int(wrong_all.sum())} câu cả {len(MODELS)} mô hình cùng sai")
    shown = P[FOCUS][wrong_all][["free_text", "true_label", "pred_label", "confidence"]]
    display(shown.sample(min(15, len(shown)), random_state=42))
else:
    print("Mới có 1 mô hình - ô này sẽ có nội dung khi có thêm PhoBERT/ViSoBERT.")

177 câu cả 3 mô hình cùng sai


,free_text,true_label,pred_label,confidence
407,"mới có anh công an "" vuốt tóc "" chị bán hàng rong mà bê bết máu luôn ấy chứ!",THÙ GHÉT,BÌNH THƯỜNG,0.8770
804,lên con cac gì nữa lộc phải là yên nghỉ chứ,BÌNH THƯỜNG,XÚC PHẠM,0.7214
2259,Cai mieng hay la cai ong cong vay,XÚC PHẠM,BÌNH THƯỜNG,0.6142
568,Thờ buổi này đi du lịch chi vậy 😀 đi để kịp đầu thai hả ha gì mà đi dữ vậy,THÙ GHÉT,BÌNH THƯỜNG,0.7172
1148,Truyền thông nó cứ nói thổi phồng giá trị của TPP thôi chứ VN đang tham gia hàng chục cái hiệp định tương tự này mà có cái mẹ gì thay đổi đâu,BÌNH THƯỜNG,XÚC PHẠM,0.9697
376,"Tìm chó thất lạc đặc điểm""cực kì thuyết phục""",XÚC PHẠM,BÌNH THƯỜNG,0.5194
1952,"Ciel Phạm thôi b ơi nói mấy nhiều ng vẫn cố chấp thôi. Bản tính ích kỷ cùng chung dòng máu VN mà ăn thua sân si. Bm ngta luôn muốn cái nào tốt đẹp nhất, tin tưởng VN điều trị thì phải đưa về VN ch...",THÙ GHÉT,BÌNH THƯỜNG,0.9852
2926,Đọc mấy dòng cuối tức muốn điên luôn,XÚC PHẠM,BÌNH THƯỜNG,0.9966
1838,Con cat ne choi hong,XÚC PHẠM,BÌNH THƯỜNG,0.9950
2274,"thề đi éo đâu cũng gặp ông long này , hay phá game vãi đái , đéo nhăt đc mồm =)))",BÌNH THƯỜNG,XÚC PHẠM,0.9030


## 5. Phân tích

### 5.1. Mất cân bằng dữ liệu

In [19]:
rows = []
for m in MODELS:
    pc = json.load(open(RESULTS_METRICS / f"{m}.json", encoding="utf-8"))["test"]["per_class"]
    for cname, v in pc.items():
        rows.append({"mô hình": m, "lớp": cname, **v})
imb = pd.DataFrame(rows)
imb["% test"] = (imb.support / imb.groupby("mô hình").support.transform("sum") * 100).round(1)
display(imb.round(4))
print("F1 theo lớp:")
display(imb.pivot(index="lớp", columns="mô hình", values="f1").reindex(list(LABELS.values())).round(4))

,mô hình,lớp,precision,recall,f1,support,% test
0,baseline,BÌNH THƯỜNG,0.9351,0.9500,0.9425,2458,82.3
1,baseline,XÚC PHẠM,0.4848,0.3980,0.4372,201,6.7
2,baseline,THÙ GHÉT,0.6347,0.6288,0.6317,326,10.9
3,phobert,BÌNH THƯỜNG,0.9522,0.9150,0.9332,2458,82.3
4,phobert,XÚC PHẠM,0.4234,0.5771,0.4884,201,6.7
5,phobert,THÙ GHÉT,0.6304,0.6748,0.6519,326,10.9
6,visobert,BÌNH THƯỜNG,0.9374,0.9325,0.9349,2458,82.3
7,visobert,XÚC PHẠM,0.4537,0.4627,0.4581,201,6.7
8,visobert,THÙ GHÉT,0.6209,0.6380,0.6293,326,10.9


F1 theo lớp:


mô hình,baseline,phobert,visobert
lớp,,,
BÌNH THƯỜNG,0.9425,0.9332,0.9349
XÚC PHẠM,0.4372,0.4884,0.4581
THÙ GHÉT,0.6317,0.6519,0.6293


In [26]:
cm = {m: np.array(R[m]["test"]["confusion_matrix"]) for m in MODELS}
rows = []
for m in MODELS:
    r1 = cm[m][1]
    rows.append({"mô hình": m, "XP→BT": r1[0], "XP→XP (đúng)": r1[1], "XP→TG": r1[2], "recall XP": round(r1[1] / r1.sum(), 3)})
display(pd.DataFrame(rows))

wrong_all = np.logical_and.reduce([~P[m].correct.values for m in MODELS])
t = P["baseline"].true_label
out = pd.DataFrame({"cả 3 mô hình cùng sai": pd.Series(wrong_all).groupby(t.values).sum(), "tổng câu": t.value_counts()}).reindex(CLASSES)
out["% cả 3 cùng sai"] = (out.iloc[:, 0] / out["tổng câu"] * 100).round(1)
display(out)

,mô hình,XP→BT,XP→XP (đúng),XP→TG,recall XP
0,baseline,74,80,47,0.398
1,phobert,56,116,29,0.577
2,visobert,63,93,45,0.463


,cả 3 mô hình cùng sai,tổng câu,% cả 3 cùng sai
BÌNH THƯỜNG,59,2458,2.4
XÚC PHẠM,60,201,29.9
THÙ GHÉT,58,326,17.8


lớp BÌNH THƯỜNG chiếm số lượng mẫu lớn nhất (82,55% dữ liệu train) nên mô hình sẽ có dự đoán thên về lớp đa số

lớp XÚC PHẠM có F1 chỉ 0,44 - 0,49 thấp nhất trong 3 lớp ở cả 3 mô hình

- lớp này ít mẫu nhất (khoảng 6,7% dữ liệu train)
- XÚC PHẠM gần BÌNH THƯỜNG (nói đùa, thân mật), gần THÙ GHÉT (công kích)
- lớp thiểu số nên dễ bị nhầm sang lớp đa số nên % sai của XÚC PHẠM lớn hơn hẳn so với 2 nhãn còn lại

### 5.2. Bias đối với các từ chỉ nhóm người hoặc danh tính
- False positive (FP): câu bình thường bị đoán nhầm thành độc hại
- Các câu BÌNH THƯỜNG có chứa từ chỉ nhóm, tính tỉ lệ đoán nhầm là độc hại (FPR) và so với FPR chung

In [20]:
IDENTITY_TERMS = {
    "chính trị": ["cộng sản", "đảng", "phản động", "dân chủ"],
    "tôn giáo": ["công giáo", "thiên chúa", "phật giáo", "hồi giáo", "tin lành"],
    "vùng miền": ["bắc kỳ", "nam kỳ", "miền bắc", "miền nam", "miền trung"],
    "quốc tịch": ["trung quốc", "hàn quốc", "nhật bản", "campuchia", "thái lan", "người mỹ"],
    "giới / xu hướng": ["gay", "đồng tính", "lgbt", "phụ nữ", "đàn bà", "đàn ông"],
}


def has_term(series, terms):
    pat = "|".join(rf"(?<!\w){re.escape(t)}(?!\w)" for t in terms)
    return series.fillna("").str.lower().str.contains(pat, regex=True)


rows = []
for m, p in P.items():
    normal = p[p.true_id == 0]
    base_fpr = (normal.pred_id != 0).mean() * 100
    for g, terms in IDENTITY_TERMS.items():
        mask = has_term(p.free_text, terms)
        sub = p[mask]
        sub_n = sub[sub.true_id == 0]
        rows.append({"mô hình": m, "nhóm": g, "số câu": int(mask.sum()),
                     "% nhãn độc hại thật": round((sub.true_id != 0).mean() * 100, 1) if len(sub) else np.nan,
                     "số câu BÌNH THƯỜNG": len(sub_n),
                     "FPR nhóm (%)": round((sub_n.pred_id != 0).mean() * 100, 1) if len(sub_n) else np.nan,
                     "FPR chung (%)": round(base_fpr, 1)})
display(pd.DataFrame(rows))

,mô hình,nhóm,số câu,% nhãn độc hại thật,số câu BÌNH THƯỜNG,FPR nhóm (%),FPR chung (%)
0,baseline,chính trị,35,51.4,17,35.3,5.0
1,baseline,tôn giáo,1,100.0,0,NaN,5.0
2,baseline,vùng miền,5,60.0,2,50.0,5.0
3,baseline,quốc tịch,13,61.5,5,0.0,5.0
4,baseline,giới / xu hướng,18,50.0,9,0.0,5.0
5,phobert,chính trị,35,51.4,17,52.9,8.5
6,phobert,tôn giáo,1,100.0,0,NaN,8.5
7,phobert,vùng miền,5,60.0,2,50.0,8.5
8,phobert,quốc tịch,13,61.5,5,20.0,8.5
9,phobert,giới / xu hướng,18,50.0,9,11.1,8.5


### 5.3. Viết tắt, nói lái, chèn ký tự, ẩn ý

In [27]:
view = P["visobert"].loc[wrong_all, ["free_text", "true_label", "pred_label", "confidence"]]
print(f"{len(view)} câu cả 3 mô hình cùng sai ({len(view) / len(y):.1%} test)")
print("Loại nhầm:"); print(pd.crosstab(view.true_label, view.pred_label).to_string())
e = P["visobert"]; print(f"\nViSoBERT sai với độ tin cậy > 0.95: {int(((~e.correct) & (e.confidence > 0.95)).sum())} câu")
display(view.sample(14, random_state=3).assign(free_text=lambda d: d.free_text.str.slice(0, 110)))

177 câu cả 3 mô hình cùng sai (5.9% test)
Loại nhầm:
pred_label   BÌNH THƯỜNG  THÙ GHÉT  XÚC PHẠM
true_label                                  
BÌNH THƯỜNG            0        33        26
THÙ GHÉT              40         0        18
XÚC PHẠM              40        20         0

ViSoBERT sai với độ tin cậy > 0.95: 180 câu


,free_text,true_label,pred_label,confidence
718,Hoàng Xuân Tâm mày muốn đợi Tùng ra mv thì mày phải ráng chờ anh GD tung sản phẩm đã khi đó Tùng mới có ý tưởn,THÙ GHÉT,BÌNH THƯỜNG,0.9939
903,Trần Hiếu Cút mẹ mày luôn đi. 🤣🤣🤣,BÌNH THƯỜNG,XÚC PHẠM,0.9874
658,Thiệt muốn lạy. Emm thấy mng ở khu cách li tập thể đc phục vụ rất tận tình chu đáo. Có cả nghệ sĩ ca sĩ v mà m,BÌNH THƯỜNG,THÙ GHÉT,0.9702
1838,Con cat ne choi hong,XÚC PHẠM,BÌNH THƯỜNG,0.9950
2677,Đêm phia mà Cười k nhặt đc mồm :) rách cmn.cả miệng,BÌNH THƯỜNG,XÚC PHẠM,0.8195
472,"em nghĩ sặc khoai chết thì hợp lý, tổ cha bọn nó",XÚC PHẠM,THÙ GHÉT,0.9924
809,Phúc Báo làm kiểu ta đây giàu đi chơi du lịch qua mấy vùng dịch chơi ấy mà,XÚC PHẠM,BÌNH THƯỜNG,0.9944
1259,Dương Hoàng xin link để đạp vào mõm chúng nó bạn,XÚC PHẠM,BÌNH THƯỜNG,0.8450
1367,"giác ngộ đi , tp HCM nhé , bác giết SG rồi 😆😆😆😆",THÙ GHÉT,BÌNH THƯỜNG,0.9968
701,Mỹ Vân liều như chó dị đó,BÌNH THƯỜNG,THÙ GHÉT,0.9772


**Nhãn không nhất quán** Câu chứa lời chửi trực tiếp (`Cút mẹ mày luôn đi`) nhưng gán BÌNH THƯỜNG, các câu chửi trực tiếp khác thường được gán độc hại (từ quy luật chung của dữ liệu). Do nhãn gốc của dataset gây nhiễu

**Độc hại ẩn ý, không có từ nhận biết** Câu mang ý công kích/đe dọa bằng cách nói vòng hoặc mỉa mai (kèm emoji cười) bị cả ba đoán BÌNH THƯỜNG với độ tin cậy khoảng 0,99. Không có token thô tục nên cả TF-IDF lẫn Transformer đều không nhận biết được

**Từ thô tục dùng như từ cảm thán/so sánh** (`rách cmn`, `liều như chó`, `Xã hội chó má`): thô tục xuất hiện cả ở câu bình thường lẫn câu độc hại, mô hình dựa vào từ thô tục sẽ nhầm. phân biệt nằm ở *đối tượng bị nhắm tới*

**Câu không dấu / viết hoa toàn bộ.** Ví dụ `🄳🅄 🅃🅁🄴🄽🄳 🄶🄷🄴 🄷🄾🄽🄶 ?` đã đổi chữ dạng khung về chữ Latin thành `DU TREND GHE HONG ?` nhưng câu không dấu và in hoa, ưkhiến nghĩa mơ hồ

### 5.4. Trích dẫn nội dung độc hại và công kích trực tiếp

In [29]:
display(error_by_flag(QUOTE_COLS))

quoted = F[QUOTE_COLS].any(axis=1).values
q_err = P[FOCUS][quoted & ~P[FOCUS].correct.values]
print(f"[{FOCUS}] {len(q_err)} câu lỗi có trích dẫn/dẫn lời (trên {int(quoted.sum())} câu có đặc điểm này)")
display(q_err[["free_text", "true_label", "pred_label", "confidence"]].head(30))

,mô hình,đặc điểm,số câu có,% lỗi khi có,% lỗi khi không
0,baseline,trích dẫn (ngoặc kép),49,18.4,12.1
1,baseline,dẫn lời (bảo/nói là),18,16.7,12.2
2,phobert,trích dẫn (ngoặc kép),49,24.5,13.2
3,phobert,dẫn lời (bảo/nói là),18,16.7,13.4
4,visobert,trích dẫn (ngoặc kép),49,22.4,13.0
5,visobert,dẫn lời (bảo/nói là),18,27.8,13.0


[visobert] 16 câu lỗi có trích dẫn/dẫn lời (trên 66 câu có đặc điểm này)


,free_text,true_label,pred_label,confidence
280,"COCC nó đâu nào dám cho ""cắt cổ"", điều từ chỗ này sang chỗ kia thì lại sẽ ăn theo vị trí, cái máu đã chảy trong người thì cái gì ăn được là nó ăn dọng hết thôi.",BÌNH THƯỜNG,THÙ GHÉT,0.8865
376,"Tìm chó thất lạc đặc điểm""cực kì thuyết phục""",XÚC PHẠM,BÌNH THƯỜNG,0.5194
407,"mới có anh công an "" vuốt tóc "" chị bán hàng rong mà bê bết máu luôn ấy chứ!",THÙ GHÉT,BÌNH THƯỜNG,0.8770
489,Nguyễn Văn Nam cái đó gọi là ngu đấy bạn...😂😂😂 Và chắc ở đây cũng có nhìu ae giống vậy 😂😂😂,BÌNH THƯỜNG,THÙ GHÉT,0.9842
492,"Địt mẹ mất cái xe thì:"" Chúng tôi đang điều tra và giải quyết"". Bán hàng rong thì giải quyết ngay lập tức không được để vấn nạn này xảy ra. Nực cười vc",THÙ GHÉT,XÚC PHẠM,0.8559
870,"Tục ngữ VN có câu: ""Bói ra ma ,quét nhà ra rác ""bác Dần à, bác nổ vui thật trúng trật mặc kệ.",BÌNH THƯỜNG,THÙ GHÉT,0.8403
1008,"Beng beng :""))",BÌNH THƯỜNG,XÚC PHẠM,0.5641
1464,"Các DLV vào bênh bảo là tài sản khỏi cần kê khai, miễn sao có 3 củ cho tụi nó là ok",BÌNH THƯỜNG,THÙ GHÉT,0.7760
1467,Nguyễn Quốc Thy kkk. Mới tương tác có 1 lần. Bắt con chó sau nhà bóp mỏ tụi “CKTP” lại lên tóp cái sáng nay lên fan cứng. Ghê thiệt 🤣🤣,BÌNH THƯỜNG,THÙ GHÉT,0.9396
1920,"đông lào thì sơn nữa xe phía sau là xe tăng ! 2 bên hông ghi rất to hàng chữ "" người có uy tín "" để bà con né cho xa !",THÙ GHÉT,BÌNH THƯỜNG,0.9773
